# 02 - Cross-State Coverage Analysis

Build a state x metric completeness matrix, identify coverage gaps, and visualize patterns across jurisdictions.

**Toolkit modules used**: `cj_data_quality.coverage`, `cj_data_quality.visualization`

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import matplotlib.pyplot as plt

from cj_data_quality.notebook_utils import setup_notebook
from cj_data_quality.sample_data import generate_and_load
from cj_data_quality.coverage.coverage_matrix import (
    build_coverage_matrix,
    identify_coverage_gaps,
    summarize_coverage,
)
from cj_data_quality.visualization.heatmaps import plot_coverage_heatmap

setup_notebook()
print("Setup complete.")

## Load Data

In [ ]:
df = generate_and_load(n_records=50000, seed=42)
print(f"Shape: {df.shape}")
print(f"States: {df['state_code'].nunique()}")

## Build Coverage Matrix

Compute completeness (1 - null rate) for key metrics across all states.

In [ ]:
metric_cols = [
    "admission_date", "release_date", "offense_date", "sentence_date",
    "race", "ethnicity", "sex", "age",
    "total_population", "admission_count", "release_count",
]

coverage_matrix = build_coverage_matrix(df, state_col="state_code", metric_cols=metric_cols)
print(f"Matrix shape: {coverage_matrix.shape}")
coverage_matrix.head(10)

## Coverage Heatmap

Red = low completeness, green = high completeness.

In [ ]:
fig = plot_coverage_heatmap(
    coverage_matrix,
    title="Cross-State Data Coverage: Corrections Reporting",
    figsize=(16, 20),
)
plt.show()

## Identify Coverage Gaps

Flag any state-metric pair with completeness below 80%.

In [ ]:
gaps = identify_coverage_gaps(coverage_matrix, threshold=0.80)
print(f"Total gaps found: {len(gaps)}")

if gaps:
    gap_df = pd.DataFrame([
        {"State": g.state_code, "Metric": g.metric_name, "Completeness": f"{g.completeness:.1%}"}
        for g in gaps[:20]
    ])
    gap_df

## Coverage Summary by Metric

In [ ]:
summary = summarize_coverage(coverage_matrix)
summary_df = pd.DataFrame(
    [{"Metric": k, "Mean Completeness": f"{v:.1%}"} for k, v in sorted(summary.items(), key=lambda x: x[1])]
)
summary_df

## Key Findings

1. **Demographic fields** (race, ethnicity) show the most variation in completeness across states
2. **Small states** (AK, WY, MT, SD, ND) tend to have worse coverage across all metrics
3. **Population metrics** are generally well-reported but some states have gaps in reporting periods
4. **Date fields** have moderate completeness, with offense_date and sentence_date most affected

This coverage matrix would be invaluable for Recidiviz's state partner onboarding — identifying which data fields need attention before analysis can begin.